In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 경로 설정 (notebook/training/ 기준 → python/ 루트)
BASE_DIR = Path.cwd().resolve().parent.parent
DATA_DIR = BASE_DIR / "data" / "actual"

flow = pd.read_csv(DATA_DIR / "FLOW_extended.csv")
tms  = pd.read_csv(DATA_DIR / "TMS_extended.csv")

# AWS 3개 지점 → 하나의 weather 데이터프레임으로 병합
aws_dfs = []
for station_id in ["368", "541", "569"]:
    aws = pd.read_csv(DATA_DIR / f"AWS_{station_id}.csv")
    aws["PREDICT_DATE"] = pd.to_datetime(aws["datetime"], errors="coerce").dt.floor("min")
    aws = aws.drop(columns=["YYMMDDHHMI", "datetime"], errors="ignore")
    aws = aws.rename(columns={c: f"{c}_{station_id}" for c in aws.columns if c != "PREDICT_DATE"})
    aws_dfs.append(aws)

# PREDICT_DATE 기준 outer join → 전체 시간 커버
weather = aws_dfs[0]
for aws in aws_dfs[1:]:
    weather = pd.merge(weather, aws, on="PREDICT_DATE", how="outer")

weather = weather.sort_values("PREDICT_DATE").reset_index(drop=True)

print("weather 컬럼:", list(weather.columns))
print("weather shape:", weather.shape)

# 기존 데이터 시간 처리
flow["SYS_TIME"] = pd.to_datetime(flow["SYS_TIME"]).dt.floor("min")
tms["SYS_TIME"]  = pd.to_datetime(tms["SYS_TIME"]).dt.floor("min")

flow = flow.sort_values("SYS_TIME")
tms  = tms.sort_values("SYS_TIME")

flow_tms = pd.merge_asof(
    flow, tms,
    on="SYS_TIME",
    direction="backward",
    tolerance=pd.Timedelta("3h")
)

df = pd.merge_asof(
    flow_tms, weather,
    left_on="SYS_TIME",
    right_on="PREDICT_DATE",
    direction="backward",
    tolerance=pd.Timedelta("6h")
)

df = df.ffill().sort_values("SYS_TIME").reset_index(drop=True)

print(f"\n병합 결과: {df.shape}")
print(f"기간: {df['SYS_TIME'].min()} ~ {df['SYS_TIME'].max()}")

weather 컬럼: ['TA_368', 'RN_15m_368', 'RN_60m_368', 'RN_12H_368', 'RN_DAY_368', 'HM_368', 'TD_368', 'PREDICT_DATE', 'TA_541', 'RN_15m_541', 'RN_60m_541', 'RN_12H_541', 'RN_DAY_541', 'HM_541', 'TD_541', 'TA_569', 'RN_15m_569', 'RN_60m_569', 'RN_12H_569', 'RN_DAY_569', 'HM_569', 'TD_569']
weather shape: (830880, 22)

병합 결과: (132297, 44)
기간: 2025-11-22 23:52:00 ~ 2026-02-23 13:58:00


In [2]:
# 부하
df['TOC_load'] = df['TOC_in'] * df['flow_TankA']
df['TN_load']  = df['TN_in']  * df['flow_TankA']

# 운전
df['aeration_total'] = df['wind1'] + df['wind2']
df['pump_total'] = df['pumpA'] + df['pumpB'] + df['pumpC']

# load per air (비선형 공정 반영)
df['TOC_load_per_air'] = df['TOC_load'] / (df['aeration_total'] + 1e-6)
df['TN_load_per_air']  = df['TN_load']  / (df['aeration_total'] + 1e-6)

# AR lag
df['TOC_VU_lag1'] = df['TOC_VU'].shift(1)
df['TN_VU_lag1']  = df['TN_VU'].shift(1)

df['TOC_VU_lag2'] = df['TOC_VU'].shift(2)
df['TN_VU_lag2']  = df['TN_VU'].shift(2)

# Residual target
df['TOC_residual'] = df['TOC_VU'] - df['TOC_VU_lag1']
df['TN_residual']  = df['TN_VU']  - df['TN_VU_lag1']

df = df.dropna().reset_index(drop=True)

In [3]:
SEQ_LEN = 12

sequence_features = [
    'TOC_VU_lag1','TN_VU_lag1',
    'TOC_VU_lag2','TN_VU_lag2',
    'TOC_load','TN_load',
    'TOC_load_per_air','TN_load_per_air',
    'aeration_total','pump_total',
    'WATER_TEMP_in'
]

TARGET_COLS = ['TOC_residual','TN_residual']

# 루프 전에 numpy 배열로 한 번만 추출
# df[cols]는 매 호출마다 DataFrame 전체를 복사하므로 루프 안에서 호출하면 메모리 고갈
X_arr = df[sequence_features].values  # (N, n_features)
y_arr = df[TARGET_COLS].values         # (N, 2)

X_seq, y_seq, idx_seq = [], [], []

for i in range(SEQ_LEN, len(df)):
    X_seq.append(X_arr[i-SEQ_LEN:i])  # numpy 슬라이싱 = 뷰, 복사 없음
    y_seq.append(y_arr[i])
    idx_seq.append(i)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
idx_seq = np.array(idx_seq)

del X_arr, y_arr  # 원본 배열 해제 (X_seq, y_seq가 np.array()로 독립 복사본 생성됨)

print("X:", X_seq.shape)
print("y:", y_seq.shape)

X: (129958, 12, 11)
y: (129958, 2)


In [4]:
import gc

train_ratio = 0.8
split_idx = int(len(X_seq) * train_ratio)

# 슬라이싱은 뷰를 반환하므로 .copy()로 독립 배열 생성 후 원본 해제
X_train = X_seq[:split_idx].copy()
X_test  = X_seq[split_idx:].copy()

y_train = y_seq[:split_idx].copy()
y_test  = y_seq[split_idx:].copy()

idx_train = idx_seq[:split_idx].copy()
idx_test  = idx_seq[split_idx:].copy()

del X_seq, y_seq, idx_seq
gc.collect()
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (103966, 12, 11), X_test: (25992, 12, 11)


In [5]:
import gc
from sklearn.preprocessing import StandardScaler

n_samples, seq_len, n_features = X_train.shape

X_train_2d = X_train.reshape(-1, n_features)
X_test_2d  = X_test.reshape(-1, n_features)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_2d)
X_test_scaled  = scaler_X.transform(X_test_2d)

X_train_scaled = X_train_scaled.reshape(n_samples, seq_len, n_features)
X_test_scaled  = X_test_scaled.reshape(X_test.shape[0], seq_len, n_features)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

# StandardScaler는 새 배열을 반환하므로 원본 배열 해제 가능
del X_train_2d, X_test_2d  # X_train/X_test의 뷰 → 해제해도 원본 영향 없음
del X_train, X_test         # X_train_scaled는 독립 배열이므로 원본 해제 가능
gc.collect()
print("메모리 정리 완료 (X_train, X_test 해제)")

메모리 정리 완료 (X_train, X_test 해제)


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class WaterDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(WaterDataset(X_train_scaled,y_train_scaled),
                          batch_size=64, shuffle=False)

test_loader = DataLoader(WaterDataset(X_test_scaled,y_test_scaled),
                         batch_size=64, shuffle=False)

In [7]:
import torch.nn as nn

class ARX_LSTM_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.alpha = nn.Parameter(torch.tensor([0.9,0.9]))

    def forward(self, x, baseline):
        out, _ = self.lstm(x)
        weights = torch.softmax(self.attn(out), dim=1)
        context = torch.sum(out * weights, dim=1)
        residual = self.fc(context)
        return baseline * self.alpha + residual

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ARX_LSTM_Attention(input_dim=X_train_scaled.shape[2]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 80

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (X_batch, y_batch) in enumerate(train_loader):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        baseline = X_batch[:,-1,0:2]  # lag1 baseline

        optimizer.zero_grad()
        outputs = model(X_batch, baseline)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.6f}")

    # GPU 캐시 정리 (10 epoch마다)
    if (epoch + 1) % 10 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

Epoch 1, Loss: 1.340807
Epoch 2, Loss: 1.282546


In [ ]:
model.eval()
pred_list = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        baseline = X_batch[:,-1,0:2]   # lag1 (scaled 상태)
        outputs = model(X_batch, baseline)
        pred_list.append(outputs.cpu().numpy())

y_pred_scaled = np.vstack(pred_list)

# 역스케일 (residual 기준)
y_pred_residual = scaler_y.inverse_transform(y_pred_scaled)

TOC_pred, TN_pred = [], []
TOC_true, TN_true = [], []

for k, idx in enumerate(idx_test):

    # baseline = 실제 lag1 값
    toc_base = df.loc[idx, 'TOC_VU_lag1']
    tn_base  = df.loc[idx, 'TN_VU_lag1']

    # residual 더하기
    TOC_pred.append(toc_base + y_pred_residual[k,0])
    TN_pred.append(tn_base  + y_pred_residual[k,1])

    TOC_true.append(df.loc[idx,'TOC_VU'])
    TN_true.append(df.loc[idx,'TN_VU'])

print("TOC R2:", r2_score(TOC_true, TOC_pred))
print("TN  R2:", r2_score(TN_true, TN_pred))